Traitement des images

In [ ]:
from PIL import Image
import os
import numpy as np
from skimage.filters import threshold_otsu

In [ ]:
# Fonction pour cropper 30 pixels de chaque côté
def crop_image(input_path, output_path):
    img = Image.open(input_path)
    width, height = img.size
    left = 30
    top = 30
    right = width - 30
    bottom = height - 30
    cropped_img = img.crop((left, top, right, bottom))
    cropped_img.save(output_path)

# Fonction pour seuiller le fond et le rendre noir avec Otsu
def threshold_background(img):
    arr = np.array(img)
    seuil = threshold_otsu(arr)
    arr[arr < seuil] = 0
    return Image.fromarray(arr)

# Fonction pour recadrer autour du corps humain (zone non noire)
def crop_to_body(img):
    arr = np.array(img)
    mask = arr > 0
    coords = np.argwhere(mask)
    if coords.size == 0:
        return img  # rien à cropper
    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0) + 1
    cropped_arr = arr[y0:y1, x0:x1]
    return Image.fromarray(cropped_arr)

# Exemple d'utilisation pour un dossier d'images
input_folder = 'images_raw/'  # à adapter
output_folder = 'images_cropped/'  # à adapter
os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        crop_image(
            os.path.join(input_folder, filename),
            os.path.join(output_folder, filename)
        )

# Exemple d'utilisation
input_folder = 'images_cropped/'  # à adapter
output_folder = 'images_normalized/'  # à adapter
os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        img = Image.open(os.path.join(input_folder, filename)).convert('L')
        img_thresh = threshold_background(img)
        img_body = crop_to_body(img_thresh)
        img_body.save(os.path.join(output_folder, filename))